In [ ]:
# Deploy the Date-Only Model as an Azure ML Web Service
# I don't access :  Add Role Assignment is disabled
from azureml.core import Workspace, Model, Environment
from azureml.core.webservice import AciWebservice
from azureml.core.model import InferenceConfig
from azureml.core.conda_dependencies import CondaDependencies

# Connect to workspace
ws = Workspace.from_config()

# Get the registered model
model = Model(ws, name='blanket_usage_date_only_model')
features_model = Model(ws, name='blanket_date_only_features')

# Create environment
env = Environment(name='blanket-prediction-env')
conda_dep = CondaDependencies()
conda_dep.add_conda_package('scikit-learn==1.0.2')
conda_dep.add_conda_package('pandas')
conda_dep.add_conda_package('numpy')
conda_dep.add_pip_package('azureml-defaults')
env.python.conda_dependencies = conda_dep

# Create inference config
inference_config = InferenceConfig(
    entry_script='score.py',
    environment=env
)

# Configure deployment
aci_config = AciWebservice.deploy_configuration(
    cpu_cores=1,
    memory_gb=1,
    auth_enabled=True,  # Enable key-based authentication
    description='Blanket usage prediction endpoint (date-only model)'
)

# Deploy the model
print("Deploying model to Azure Container Instance...")
service = Model.deploy(
    workspace=ws,
    name='blanket-prediction-endpoint',
    models=[model, features_model],
    inference_config=inference_config,
    deployment_config=aci_config,
    overwrite=True  # Overwrite if endpoint already exists
)

# Wait for deployment to complete
service.wait_for_deployment(show_output=True)

# Get endpoint URL and keys
print("\n" + "="*70)
print("DEPLOYMENT SUCCESSFUL!")
print("="*70)
print(f"Endpoint URL: {service.scoring_uri}")
print(f"Swagger URI: {service.swagger_uri}")

# Get authentication keys
keys = service.get_keys()
print(f"\nPrimary Key: {keys[0]}")
print(f"Secondary Key: {keys[1]}")
print("="*70)

I don't access : Add Role Assignment is disabled. So will need to Deploy to Managed Online Endpoint

In [39]:
# Deploy to Azure ML Managed Online Endpoint (Modern Approach)
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    Environment,
    CodeConfiguration
)
from azure.identity import DefaultAzureCredential
from azureml.core import Workspace

# Connect to workspace
ws = Workspace.from_config()

# Create MLClient
credential = DefaultAzureCredential()
ml_client = MLClient(
    credential=credential,
    subscription_id=ws.subscription_id,
    resource_group_name=ws.resource_group,
    workspace_name=ws.name
)

print(f"Connected to workspace: {ws.name}")
print(f"Resource Group: {ws.resource_group}")
print(f"Subscription: {ws.subscription_id}")

Connected to workspace: mlw-hhn-dev-20251205
Resource Group: rg-hhn-dev
Subscription: 0ded687b-997b-4159-acf3-9c82e7352f8c


In [40]:
# Step 1: Create or Update the Endpoint
import datetime

endpoint_name = "blanket-prediction-endpoint"

# Create endpoint
endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Blanket usage prediction using date-only features",
    auth_mode="key",
    tags={
        "model": "date_only_random_forest",
        "created": datetime.datetime.now().strftime("%Y-%m-%d")
    }
)

print(f"Creating endpoint: {endpoint_name}")
print("This may take a few minutes...")

# Create or update endpoint
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print(f"✓ Endpoint '{endpoint_name}' created successfully!")

Creating endpoint: blanket-prediction-endpoint
This may take a few minutes...
✓ Endpoint 'blanket-prediction-endpoint' created successfully!


In [42]:
# Step 2: Create Environment Configuration

# Create conda_env.yml file
conda_yaml = """name: blanket-prediction-env
channels:
  - conda-forge
dependencies:
  - python=3.8
  - pip
  - pip:
    - azureml-defaults
    - inference-schema
    - scikit-learn==1.0.2
    - pandas
    - numpy
    - joblib
"""

with open('conda_env.yml', 'w') as f:
    f.write(conda_yaml)

print("✓ Conda environment file created: conda_env.yml")

✓ Conda environment file created: conda_env.yml


In [46]:
# Step 3: Update score.py with better error handling and model loading

scoring_script = """
import json
import joblib
import pandas as pd
import numpy as np
import os
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def init():
    global model, features
    try:
        # Get model directory
        model_dir = os.getenv('AZUREML_MODEL_DIR')
        logger.info(f"Model directory: {model_dir}")
        
        # List all files in model directory to debug
        if model_dir and os.path.exists(model_dir):
            logger.info(f"Contents of model directory: {os.listdir(model_dir)}")
            for root, dirs, files in os.walk(model_dir):
                logger.info(f"Directory: {root}")
                logger.info(f"Files: {files}")
        
        # Try different possible paths for the model files
        # Path 1: Direct in model_dir
        model_path_1 = os.path.join(model_dir, 'blanket_usage_date_only_model.pkl')
        features_path_1 = os.path.join(model_dir, 'blanket_date_only_features.pkl')
        
        # Path 2: In subdirectory with model name
        model_path_2 = os.path.join(model_dir, 'blanket_usage_date_only_model', 'blanket_usage_date_only_model.pkl')
        features_path_2 = os.path.join(model_dir, 'blanket_date_only_features', 'date_only_features.pkl')
        
        # Try to load model
        if os.path.exists(model_path_1):
            logger.info(f"Loading model from: {model_path_1}")
            model = joblib.load(model_path_1)
        elif os.path.exists(model_path_2):
            logger.info(f"Loading model from: {model_path_2}")
            model = joblib.load(model_path_2)
        else:
            raise FileNotFoundError(f"Model file not found at {model_path_1} or {model_path_2}")
        
        # Try to load features
        if os.path.exists(features_path_1):
            logger.info(f"Loading features from: {features_path_1}")
            features = joblib.load(features_path_1)
        elif os.path.exists(features_path_2):
            logger.info(f"Loading features from: {features_path_2}")
            features = joblib.load(features_path_2)
        else:
            raise FileNotFoundError(f"Features file not found at {features_path_1} or {features_path_2}")
        
        logger.info("Model and features loaded successfully!")
        logger.info(f"Features: {features}")
        logger.info(f"Model type: {type(model)}")
        
    except Exception as e:
        logger.error(f"Error in init: {str(e)}", exc_info=True)
        raise

def run(raw_data):
    try:
        logger.info(f"Received data: {raw_data}")
        
        # Parse input data
        data = json.loads(raw_data)
        
        # Handle both formats: {'data': [...]} or direct list
        if isinstance(data, dict) and 'data' in data:
            data = data['data']
        
        # Convert to DataFrame
        df = pd.DataFrame(data)
        logger.info(f"DataFrame shape: {df.shape}")
        logger.info(f"DataFrame columns: {df.columns.tolist()}")
        
        # Make predictions
        predictions = model.predict(df[features])
        predictions = predictions.round().astype(int)
        predictions = np.clip(predictions, 0, None)  # Use numpy clip
        
        logger.info(f"Predictions: {predictions.tolist()}")
        
        # Return results
        return json.dumps({
            'predictions': predictions.tolist(),
            'status': 'success'
        })
    except Exception as e:
        error_msg = f"Error in run: {str(e)}"
        logger.error(error_msg, exc_info=True)
        return json.dumps({
            'error': error_msg,
            'status': 'failed'
        })
"""

# Save updated scoring script
with open('score.py', 'w') as f:
    f.write(scoring_script)

print("✓ Updated scoring script created: score.py")

✓ Updated scoring script created: score.py


In [47]:
# Step 4: Create Environment
from azure.ai.ml.entities import Environment

env = Environment(
    name="blanket-prediction-env",
    description="Environment for blanket prediction model",
    conda_file="conda_env.yml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"
)

print("✓ Environment configuration created")

✓ Environment configuration created


In [49]:
# Step 5: Create Deployment with both models
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

# Reference both models that were registered
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="blanket-prediction-endpoint",
    model="blanket_usage_date_only_model:1",  # Primary model
    environment=env,
    code_configuration=CodeConfiguration(
        code=".",
        scoring_script="score.py"
    ),
    instance_type="Standard_DS3_v2",
    instance_count=1,
    request_settings={
        "request_timeout_ms": 30000,
        "max_concurrent_requests_per_instance": 1
    }
)

print("Creating deployment...")
print("This will take 10-15 minutes. Please wait...")

try:
    # Deploy
    ml_client.online_deployments.begin_create_or_update(deployment).result()
    print("✓ Deployment 'blue' created successfully!")
except Exception as e:
    print(f"Deployment failed: {str(e)}")
    print("\nChecking deployment logs...")
    # Try to get logs
    try:
        logs = ml_client.online_deployments.get_logs(
            name="blue",
            endpoint_name="blanket-prediction-endpoint",
            lines=100
        )
        print("Deployment logs:")
        print(logs)
    except:
        print("Could not retrieve logs")

Check: endpoint blanket-prediction-endpoint exists


Creating deployment...
This will take 10-15 minutes. Please wait...


Uploading AzurePortal (3.35 MBs): 100%|██████████| 3345982/3345982 [00:01<00:00, 2381673.06it/s]


ActivityCompleted: Activity=OnlineDeployment.BeginCreateOrUpdate, HowEnded=Failure, Duration=14356.49 [ms], Exception=AttributeError, ErrorCategory=Unknown, ErrorMessage=Got error AttributeError: ''dict' object has no attribute '_to_rest_object'' while calling OnlineDeployment.BeginCreateOrUpdate


Deployment failed: 'dict' object has no attribute '_to_rest_object'

Checking deployment logs...
Deployment logs:
Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: InProgress

Container events:
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-01-13T23:15:36.317534Z, Message: Start downloading models
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-01-13T23:15:36.368948Z, Message: Start pulling container image
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-01-13T23:15:40.351846Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-01-13T23:15:40.351846Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-01-13T23:15:40.409412Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-01-13T23:15:40.476807Z, Message: Started container inference-server

Container logs:
Liveness Pr

In [50]:
# First, check deployment status and delete if needed
try:
    deployment_status = ml_client.online_deployments.get(
        name="blue",
        endpoint_name="blanket-prediction-endpoint"
    )
    print(f"Current deployment status: {deployment_status.provisioning_state}")
    
    # If it's failed or stuck, delete it
    if deployment_status.provisioning_state in ["Failed", "Canceled"]:
        print("Deleting failed deployment...")
        ml_client.online_deployments.begin_delete(
            name="blue",
            endpoint_name="blanket-prediction-endpoint"
        ).result()
        print("✓ Deleted failed deployment")
except Exception as e:
    print(f"No existing deployment or error checking: {e}")

Current deployment status: Failed
Deleting failed deployment...
✓ Deleted failed deployment


In [ ]:
# From previous step
# Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints
# Step 6: Set Traffic to 100% for this deployment
endpoint = ml_client.online_endpoints.get("blanket-prediction-endpoint")
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print("✓ Traffic set to 100% for 'blue' deployment")

In [ ]:
# Step 7: Get Endpoint Details and Keys
endpoint = ml_client.online_endpoints.get("blanket-prediction-endpoint")
keys = ml_client.online_endpoints.get_keys("blanket-prediction-endpoint")

print("\n" + "="*70)
print("DEPLOYMENT SUCCESSFUL!")
print("="*70)
print(f"Endpoint Name: {endpoint.name}")
print(f"Endpoint URL: {endpoint.scoring_uri}")
print(f"Swagger URI: {endpoint.swagger_uri}")
print(f"\nPrimary Key: {keys.primary_key}")
print(f"Secondary Key: {keys.secondary_key}")
print("="*70)

# Save endpoint info to file
import json
endpoint_info = {
    'endpoint_name': endpoint.name,
    'endpoint_url': endpoint.scoring_uri,
    'swagger_uri': endpoint.swagger_uri,
    'primary_key': keys.primary_key,
    'secondary_key': keys.secondary_key,
    'deployment_name': 'blue',
    'created_at': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

with open('endpoint_info.json', 'w') as f:
    json.dump(endpoint_info, f, indent=4)

print("\n✓ Endpoint information saved to 'endpoint_info.json'")

In [ ]:
# Step 8: Test the Endpoint
import requests
import json
from datetime import datetime, timedelta

# Prepare test data
test_date = datetime.now() + timedelta(days=1)
test_data = {
    'data': [{
        'DayOfWeekNum': test_date.isoweekday(),
        'IsWeekend': 1 if test_date.isoweekday() >= 6 else 0,
        'Month': test_date.month,
        'DayOfMonth': test_date.day
    }]
}

# Make prediction request
headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {keys.primary_key}'
}

print("\n" + "="*70)
print("TESTING ENDPOINT")
print("="*70)
print(f"Endpoint URL: {endpoint.scoring_uri}")
print(f"Test Date: {test_date.strftime('%Y-%m-%d %A')}")
print(f"Request Data: {test_data}")

response = requests.post(
    endpoint.scoring_uri,
    data=json.dumps(test_data),
    headers=headers
)

print(f"\nResponse Status: {response.status_code}")
if response.status_code == 200:
    print(f"Prediction Result: {response.json()}")
else:
    print(f"Error: {response.text}")
print("="*70)

In [ ]:
# Step 9: Test with Multiple Days (Next 7 days)
print("\n" + "="*70)
print("TESTING WITH NEXT 7 DAYS")
print("="*70)

test_dates = []
for i in range(7):
    future_date = datetime.now() + timedelta(days=i+1)
    test_dates.append({
        'DayOfWeekNum': future_date.isoweekday(),
        'IsWeekend': 1 if future_date.isoweekday() >= 6 else 0,
        'Month': future_date.month,
        'DayOfMonth': future_date.day
    })

batch_test_data = {'data': test_dates}

response = requests.post(
    endpoint.scoring_uri,
    data=json.dumps(batch_test_data),
    headers=headers
)

if response.status_code == 200:
    result = response.json()
    predictions = result.get('predictions', [])
    
    print(f"\n{'Date':<12} {'Day':<10} {'Weekend':<10} {'Prediction':<12}")
    print("-" * 50)
    for i, pred in enumerate(predictions):
        date = datetime.now() + timedelta(days=i+1)
        is_weekend = "Yes" if date.isoweekday() >= 6 else "No"
        print(f"{date.strftime('%Y-%m-%d'):<12} {date.strftime('%A'):<10} {is_weekend:<10} {pred:<12}")
    
    print(f"\nTotal blankets needed: {sum(predictions)}")
    print(f"Average per day: {sum(predictions)/len(predictions):.1f}")
else:
    print(f"Error: {response.status_code}")
    print(response.text)

print("="*70)